# Fine tuning BERT for LLM hallucination

## 1. Getting data

In [18]:
from datasets import load_dataset

# Load a specific language (e.g., English)
dataset = load_dataset("Helsinki-NLP/mu-shroom", "en")

# Access splits
train = dataset["train_unlabeled"]
val = dataset["validation"]
test = dataset["test"]
train.__len__


<bound method Dataset.__len__ of Dataset({
    features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'wikipedia_url', 'soft_labels', 'hard_labels', 'model_output_logits', 'model_output_tokens', 'annotations'],
    num_rows: 809
})>

In [19]:
train[0]

{'id': None,
 'lang': 'EN',
 'model_input': 'Do all arthropods have antennae?',
 'model_output_text': ' Yes, all insects and arachnids (including spiders, scorpions, and ticks) have at least one antenna. ',
 'model_id': 'togethercomputer/Pythia-Chat-Base-7B',
 'wikipedia_url': None,
 'soft_labels': None,
 'hard_labels': None,
 'model_output_logits': [-2.57427001,
  5.1865358353,
  5.4173498154,
  2.3298389912,
  6.6413722038,
  3.5858919621,
  10.0333786011,
  4.0347309113,
  2.5324678421,
  0.595354557,
  2.0120403767,
  1.8963916302,
  6.8019108772,
  6.2508711815,
  4.6549735069,
  2.6137938499,
  -3.9706561565,
  7.5047488213,
  4.2792539597,
  3.0500841141,
  -0.4772415757,
  3.1624026299,
  13.7479448318,
  -1.7824890614,
  3.9432423115,
  4.9513559341,
  4.3375749588,
  14.8357839584],
 'model_output_tokens': ['ĠYes',
  ',',
  'Ġall',
  'Ġinsects',
  'Ġand',
  'Ġar',
  'ach',
  'n',
  'ids',
  'Ġ(',
  'including',
  'Ġspiders',
  ',',
  'Ġsc',
  'orp',
  'ions',
  ',',
  'Ġand',

In [4]:
# Get an example from validation set
example = dataset["validation"][0]

print(f"Language: {example['lang']}")
print(f"Input: {example['model_input']}")
print(f"Model Output: {example['model_output_text']}")
print(f"Hallucination spans: {example['hard_labels']}")

# Visualize hallucination spans
text = example["model_output_text"]
for span in example["hard_labels"]:
    start, end = span
    print(f"Hallucinated text: '{text[start:end]}'")


Language: EN
Input: What did Petra van Staveren win a gold medal for?
Model Output: Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.
Hallucination spans: [[25, 31], [45, 49], [69, 83]]
Hallucinated text: 'silver'
Hallucinated text: '2008'
Hallucinated text: 'Beijing, China'


## 2. Tokenization and Labeling Hallucinated text
`offset_mapping`
- So while tokenizing we are getting the idex for starting and ending of each and every word in our given text
- ex: Petra van Stoveren -> [(0,5), (6,9), (10,15)]

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def align_labels(text, hallucination_spans):
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True)
    offsets = enc.offset_mapping  # list of (char_start_index, char_end_index) for each word
    labels = [-10] * len(offsets)  # initialize all word labels to -10
    # Set label=1 for tokens overlapping a hallucination span
    for start, end in hallucination_spans:
        for i, (s, e) in enumerate(offsets):
            # If token intersects span
            if not (e <= start or s >= end):     
                labels[i] = 1
    return labels

# Example
text = "Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China"
char_spans = [[25, 31], [45, 49], [69, 83]]
enc = tokenizer(text, return_offsets_mapping=True)
labels = align_labels(text, char_spans)
print(list(zip(enc.tokens(), labels)))


[('[CLS]', -10), ('petra', -10), ('van', -10), ('stove', -10), ('##ren', -10), ('won', -10), ('a', -10), ('silver', 1), ('medal', -10), ('in', -10), ('the', -10), ('2008', 1), ('summer', -10), ('olympics', -10), ('in', -10), ('beijing', 1), (',', 1), ('china', 1), ('[SEP]', -10)]


## 3. DataLoader

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

class HallucinationDataset(Dataset):
    def __init__(self, texts, spans, tokenizer):
        self.texts = texts
        self.spans = spans
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        spans = self.spans[idx]  # list of [char_start, char_end]
        enc = self.tokenizer(text, 
                              return_offsets_mapping=True, 
                              truncation=True)
        labels = align_labels(text, spans)  # as above
        item = {
            "input_ids": torch.tensor(enc.input_ids),
            "attention_mask": torch.tensor(enc.attention_mask),
            "labels": torch.tensor(labels)
        }
        return item

train[0]
# dataset = HallucinationDataset(val, val_spans, tokenizer)
# # Use tokenizer.pad to collate variable-length inputs
# dataloader = DataLoader(dataset, batch_size=16, shuffle=True, 
#                         collate_fn=lambda x: tokenizer.pad(x, return_tensors="pt"))


{'id': None,
 'lang': 'EN',
 'model_input': 'Do all arthropods have antennae?',
 'model_output_text': ' Yes, all insects and arachnids (including spiders, scorpions, and ticks) have at least one antenna. ',
 'model_id': 'togethercomputer/Pythia-Chat-Base-7B',
 'wikipedia_url': None,
 'soft_labels': None,
 'hard_labels': None,
 'model_output_logits': [-2.57427001,
  5.1865358353,
  5.4173498154,
  2.3298389912,
  6.6413722038,
  3.5858919621,
  10.0333786011,
  4.0347309113,
  2.5324678421,
  0.595354557,
  2.0120403767,
  1.8963916302,
  6.8019108772,
  6.2508711815,
  4.6549735069,
  2.6137938499,
  -3.9706561565,
  7.5047488213,
  4.2792539597,
  3.0500841141,
  -0.4772415757,
  3.1624026299,
  13.7479448318,
  -1.7824890614,
  3.9432423115,
  4.9513559341,
  4.3375749588,
  14.8357839584],
 'model_output_tokens': ['ĠYes',
  ',',
  'Ġall',
  'Ġinsects',
  'Ġand',
  'Ġar',
  'ach',
  'n',
  'ids',
  'Ġ(',
  'including',
  'Ġspiders',
  ',',
  'Ġsc',
  'orp',
  'ions',
  ',',
  'Ġand',

## 4. Initializing Token Classification Model 

In [7]:
from transformers import AutoConfig, AutoModelForTokenClassification

model_name = "bert-base-uncased"  # or roberta-base, microsoft/deberta-v3-base, etc.
config = AutoConfig.from_pretrained(model_name, num_labels=2)
model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

# Training loop 

In [8]:
from torch.optim import AdamW
from transformers import get_scheduler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
num_epochs = 3
num_training_steps = num_epochs * len(dataloader)
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer,
    num_warmup_steps=500, num_training_steps=num_training_steps
)

model.train()
for epoch in range(num_epochs):
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()


NameError: name 'dataloader' is not defined